# SP・PB・APのカテゴリ別売上を月と季節で比較

## 目的とデータ

SPの平均注文額向上と、PB・APの販促仮説を考えるため、3州のカテゴリ別の注文数・GMV・注文額を時系列で確認する。入力はAthenaの[月×州×カテゴリ集計SQL](../sql/athena/20_monthly_category_sales_by_state.sql)を実行して保存した`../data/monthly_category_sales_by_state.csv`と、既存の`../data/sales_baseline_by_month_state.csv`。

比較するカテゴリは`bed_bath_table`、`health_beauty`、`watches_gifts`、`computers_accessories`。APで1注文の`computers`などは推移図から外す。元CSVには全カテゴリを残し、集計の照合に使う。

**指標の違い**

- カテゴリ注文数：そのカテゴリの商品を含む注文数。同じ注文が複数カテゴリに含まれる場合、カテゴリごとに数える。
- カテゴリGMV：そのカテゴリの商品の価格合計。送料は含まない。
- カテゴリ注文のAOV：そのカテゴリを含む注文の**注文全体のGMV**合計 ÷ カテゴリ注文数。複数カテゴリを含む注文の全額が各カテゴリに入るため、カテゴリ間では足せない。
- **平均注文額へのカテゴリ構成額**：カテゴリGMV ÷ その州・期間の全注文数。全カテゴリの構成額を足すと、その州・期間の平均注文額になる。3州のGMV規模が違うため、推移図の縦軸にはこの指標を使う。

ブラジルの四季は分析上、夏＝12～2月、秋＝3～5月、冬＝6～8月、春＝9～11月と定義する。12月は翌年の夏に含める。比較には2017年3月～2018年8月を使い、3か月がそろう6季節を作る。2016年の初期月は注文数が少なく、季節比較から外す。


In [ ]:
from pathlib import Path
import sys

# Notebookをどの作業フォルダから開いてもsrc/を読み込めるようにする。
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.project_paths import DATA_DIR, IMAGE_DIR
from src.sales_metrics import (
    monthly_category_metrics, seasonal_category_metrics, category_plot_data,
)
from src.analysis_plots import plot_category_trends, plot_sp_category_trends

df_month_category = pd.read_csv(DATA_DIR / "monthly_category_sales_by_state.csv")
df_baseline = pd.read_csv(DATA_DIR / "sales_baseline_by_month_state.csv")
print("月×州×カテゴリ:", df_month_category.shape)
display(df_month_category.head())
display(df_month_category.isna().sum())

In [ ]:
# 州全体の注文数を結合し、カテゴリGMVが月×州の基準値と一致するか確認する。
target_states = ["SP", "PB", "AP"]
monthly, state_month, reconciliation = monthly_category_metrics(
    df_month_category, df_baseline, target_states
)
display(reconciliation.tail(12).round({
    "category_gmv_sum": 2, "state_gmv": 2, "gmv_difference": 2,
}))
print("GMV差の絶対値の最大:", reconciliation["gmv_difference"].abs().max())

## 季節の作り方

月ごとの注文は一つの季節にだけ入る。したがって、季節のカテゴリ注文数・カテゴリGMV・州の注文数は、対応する3か月の**合計**を使う。AOVは月別AOVの単純平均にせず、季節全体の金額合計を注文数合計で割り直す。


In [ ]:
# 期間を決め、3か月の分子・分母を合算してから季節指標を計算する。
analysis_start = pd.Timestamp("2017-03-01")
analysis_end = pd.Timestamp("2018-08-01")
monthly_period, state_month_period, seasonal, season_state = seasonal_category_metrics(
    monthly, state_month, analysis_start, analysis_end
)
display(season_state.sort_values(
    ["season_start", "customer_state"]
).round({"state_gmv": 2}))

## 月別・季節別の比較表

表では、カテゴリ注文数、カテゴリGMV、カテゴリ注文のAOV、平均注文額への構成額を確認する。APは注文数が少ないため、特に月別の値は数件の注文で大きく変わりうる。グラフを読む際にも、同じ期間の州全体の注文数を確認する。


In [4]:
focus_categories = [
    "bed_bath_table",
    "health_beauty",
    "watches_gifts",
    "computers_accessories",
]
columns_to_show = [
    "customer_state", "product_category", "state_orders",
    "category_order_count", "category_gmv",
    "category_order_aov", "aov_part",
]

# まず直近の月と季節を表示する。期間を変更しても同じ列で比較できる。
latest_month = monthly_period["month_start"].max()
latest_season = seasonal["season_start"].max()
print("月別:", latest_month.strftime("%Y-%m"))
display(
    monthly_period.loc[
        (monthly_period["month_start"] == latest_month)
        & monthly_period["product_category"].isin(focus_categories),
        columns_to_show,
    ].sort_values(["product_category", "customer_state"]).round(2)
)
print("季節別:", latest_season.strftime("%Y-%m"))
display(
    seasonal.loc[
        (seasonal["season_start"] == latest_season)
        & seasonal["product_category"].isin(focus_categories),
        columns_to_show,
    ].sort_values(["product_category", "customer_state"]).round(2)
)


月別: 2018-08


,customer_state,product_category,state_orders,category_order_count,category_gmv,category_order_aov,aov_part
1425,SP,bed_bath_table,3164,320,37305.79,117.66,11.79
1410,AP,computers_accessories,2,1,99.00,99.00,49.50
1417,PB,computers_accessories,28,4,638.70,159.68,22.81
1430,SP,computers_accessories,3164,179,17369.38,97.26,5.49
1416,PB,health_beauty,28,4,770.65,192.66,27.52
1424,SP,health_beauty,3164,370,49676.60,135.08,15.70
1413,PB,watches_gifts,28,3,1194.91,398.30,42.68
1427,SP,watches_gifts,3164,168,26245.44,156.81,8.30


季節別: 2018-06


,customer_state,product_category,state_orders,category_order_count,category_gmv,category_order_aov,aov_part
506,PB,bed_bath_table,107,4,281.69,70.42,2.63
539,SP,bed_bath_table,8617,932,102152.61,110.86,11.85
497,AP,computers_accessories,10,1,99.00,99.00,9.90
509,PB,computers_accessories,107,12,1434.80,119.57,13.41
546,SP,computers_accessories,8617,484,52168.36,107.99,6.05
500,AP,health_beauty,10,1,122.99,122.99,12.30
517,PB,health_beauty,107,18,4193.32,232.96,39.19
573,SP,health_beauty,8617,1049,126922.31,122.27,14.73
503,AP,watches_gifts,10,1,78.00,78.00,7.80
531,PB,watches_gifts,107,11,6246.81,567.89,58.38


In [5]:
# 表を詳しく読むカテゴリを1つ選ぶ。名前を変えて同じ集計を確認できる。
selected_category = "health_beauty"

# カテゴリの注文がなかった月・州は、この表には行がない。
# 後の推移図では、州の注文があれば構成額0として表示する。
monthly_detail = monthly_period.loc[
    monthly_period["product_category"] == selected_category,
    ["purchase_month", "customer_state", "state_orders",
     "category_order_count", "category_gmv",
     "category_order_aov", "aov_part"],
].sort_values(["purchase_month", "customer_state"])
seasonal_detail = seasonal.loc[
    seasonal["product_category"] == selected_category,
    ["season_label", "season_start", "customer_state", "state_orders",
     "category_order_count", "category_gmv",
     "category_order_aov", "aov_part"],
].sort_values(["season_start", "customer_state"])

print("月別:", selected_category)
display(monthly_detail.round(2))
print("季節別:", selected_category)
display(seasonal_detail.drop(columns="season_start").round(2))


月別: health_beauty


,purchase_month,customer_state,state_orders,category_order_count,category_gmv,category_order_aov,aov_part
124,2017-03,PB,16,2,711.90,355.95,44.49
138,2017-03,SP,966,71,9434.53,132.88,9.77
183,2017-04,PB,20,2,496.49,248.24,24.82
198,2017-04,SP,873,62,8465.85,137.09,9.70
246,2017-05,AP,4,2,245.90,122.95,61.48
249,2017-05,PB,17,3,424.99,141.66,25.00
263,2017-05,SP,1363,107,11745.98,109.78,8.62
313,2017-06,PB,22,3,1658.00,552.67,75.36
334,2017-06,SP,1284,90,10050.44,112.16,7.83
375,2017-07,AP,1,1,259.90,259.90,259.90


季節別: health_beauty


,season_label,customer_state,state_orders,category_order_count,category_gmv,category_order_aov,aov_part
5,2017 Autumn,AP,7,2,245.90,122.95,35.13
19,2017 Autumn,PB,53,7,1633.38,233.34,30.82
66,2017 Autumn,SP,3202,240,29646.36,123.67,9.26
93,2017 Winter,AP,6,1,259.90,259.90,43.32
108,2017 Winter,PB,64,6,1878.90,313.15,29.36
156,2017 Winter,SP,4492,336,32975.36,98.39,7.34
187,2017 Spring,AP,9,3,234.89,78.30,26.10
204,2017 Spring,PB,80,14,2913.51,208.11,36.42
261,2017 Spring,SP,6198,439,60296.28,137.71,9.73
310,2018 Summer,PB,103,11,1577.15,143.38,15.31


## 州ごとに色を固定して月別・季節別を比較する

月別と季節別をそれぞれ**1枚の図**にまとめ、各カテゴリの小図でSP・PB・APを同じ目盛り上に描く。SPは青、PBは橙、APは緑に固定し、線種と記号でも州を区別する。**縦軸の範囲はカテゴリごとに異なる**ため、小図をまたいで線の高さを直接比較しない。月×州でそのカテゴリの注文がなければ、平均注文額への構成額は0。州全体の注文もなければ値は表示しない。


In [ ]:
# 月別と季節別の州×カテゴリをそろえ、同じ州を同じ色で描く。
monthly_plot = category_plot_data(
    state_month_period.rename(columns={"month_start": "period_start"}),
    monthly_period.rename(columns={"month_start": "period_start"}),
    "period_start", pd.date_range(analysis_start, analysis_end, freq="MS"),
    target_states, focus_categories,
)
seasonal_plot = category_plot_data(
    season_state, seasonal, "season_start",
    pd.DatetimeIndex(sorted(season_state["season_start"].unique())),
    target_states, focus_categories,
)
plot_category_trends(
    monthly_plot, "period_start", focus_categories, target_states,
    "Monthly category contribution to AOV: SP, PB, AP",
    IMAGE_DIR / "Monthly_Category_AOV_Contribution_SP_PB_AP.png",
)
plot_category_trends(
    seasonal_plot, "season_start", focus_categories, target_states,
    "Seasonal category contribution to AOV: SP, PB, AP",
    IMAGE_DIR / "Seasonal_Category_AOV_Contribution_SP_PB_AP.png",
)

## SP州のカテゴリ構成額を拡大して見る

APは月の州全体の注文数が少なく、1件の高額注文で縦軸が大きく伸びる。SPの4カテゴリの推移を読みやすくするため、SPだけを月別と季節別で並べる。この図では州の色である青に統一し、カテゴリは線種と記号で区別する。ここでも縦軸は「カテゴリGMV ÷ 州全体の注文数」であり、カテゴリを含む注文だけのAOVではない。


In [ ]:
# SPを拡大して表示。州色は青、カテゴリは線種と記号で区別する。
plot_sp_category_trends(
    monthly_plot, seasonal_plot, focus_categories,
    IMAGE_DIR / "SP_Category_AOV_Contribution_Monthly_Seasonal.png",
)

## 図の次に確認すること

図で差や変化が見つかっても、季節性や施策効果とすぐに判断しない。各点のカテゴリ注文数と州全体の注文数、商品・販売者の構成、年ごとの再現性を確認する。PB・APは少数注文による振れが大きいため、SPへの転用可能性も別途検討する。販促費・原価がない限り、利益の増加は計算できない。
